# Phi-4-mini LoRA Evaluation Only

شغّل السيلات بالترتيب من فوق لتحت.  
النوتبوك دي **مش هتدرّب**، هي بس هتعمل:

1. Load dataset  
2. Find saved LoRA adapter automatically  
3. Load base Phi-4-mini  
4. Load LoRA  
5. Run inference  
6. Evaluate  
7. Save CSV + metrics


In [1]:
# CELL 1 — Optional installs/check only
# لو كل المكتبات موجودة سيبها زي ما هي. لو ناقص حاجة فك الكومنت وشغل التثبيت.
# !pip install -q transformers==4.49.0 accelerate==1.3.0 bitsandbytes==0.45.3 peft==0.14.0 sentence-transformers rapidfuzz tqdm tabulate

import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print('Environment variables set.')


Environment variables set.


In [2]:
# CELL 2 — Imports + main settings
import os, gc, re, json, time, random, zipfile, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from rapidfuzz import fuzz as rfuzz

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer

RUN_TYPE = "Summarized Only"
MODEL_ID = "microsoft/Phi-4-mini-instruct"

# غير DATA_PATH فقط لو اسم الداتا عندك مختلف
DATA_PATH = Path("/kaggle/input/datasets/belalhesham/koskoskos/qa_data_answer_focused_summarized.json")

# ده المكان اللي المفروض الموديل موجود فيه لو التدريب خلص في نفس Session
WORKING_ADAPTER_DIR = Path("/kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG")

OUTPUT_DIR = Path("/kaggle/working/PHI4_EVAL_ONLY")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_4BIT = True
SEED = 42
TEST_RATIO = 0.20
SAMPLE_SIZE = None  # خليها None عشان يستخدم كل الداتا
MAX_NEW_TOKENS = 64
SEM_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

# dummy training vars عشان السيف/الملخص ميعملش Error
train_time = 0
trainable_params = 0
NUM_EPOCHS = 0
LEARNING_RATE = 0
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Setup ready.')


Setup ready.


In [3]:
# CELL 3 — Find the trained LoRA adapter
# الأول بيدور في /kaggle/working عشان تكمل من آخر تدريب.
# لو مش لاقيه، يدور في /kaggle/input لو انت رافعه كـ Dataset.

def find_adapter_dir():
    candidates = []

    # 1) exact working folder
    if (WORKING_ADAPTER_DIR / "adapter_config.json").exists():
        candidates.append(WORKING_ADAPTER_DIR)

    # 2) checkpoints inside working folder
    if WORKING_ADAPTER_DIR.exists():
        for p in sorted(WORKING_ADAPTER_DIR.rglob("adapter_config.json")):
            candidates.append(p.parent)

    # 3) any adapter in /kaggle/working
    for p in sorted(Path("/kaggle/working").rglob("adapter_config.json")):
        candidates.append(p.parent)

    # 4) fallback: any adapter in /kaggle/input
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in sorted(input_root.rglob("adapter_config.json")):
            candidates.append(p.parent)

    # remove duplicates
    uniq = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            uniq.append(c)
            seen.add(s)

    if not uniq:
        raise FileNotFoundError("No adapter_config.json found in /kaggle/working or /kaggle/input")

    print("Found adapters:")
    for i, c in enumerate(uniq):
        print(f"[{i}] {c}")

    # Prefer exact working dir if exists, otherwise prefer latest/highest checkpoint in working, otherwise first input adapter
    working = [c for c in uniq if str(c).startswith('/kaggle/working')]
    if working:
        def ckpt_num(path):
            m = re.search(r'checkpoint-(\d+)', str(path))
            return int(m.group(1)) if m else 10**12
        return sorted(working, key=ckpt_num)[-1]
    return uniq[0]

MODEL_DIR = find_adapter_dir()
zip_path = OUTPUT_DIR / f"{MODEL_DIR.name}.zip"
print("\nUSING MODEL_DIR:", MODEL_DIR)
print("adapter_config exists:", (MODEL_DIR / 'adapter_config.json').exists())


Found adapters:
[0] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora
[1] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora/checkpoint-1000
[2] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora/checkpoint-2000
[3] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/phi4mini_summarized_only_lora/phi4mini_summarized_only_lora
[4] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/phi4mini_summarized_only_lora/phi4mini_summarized_only_lora/checkpoint-1000
[5] /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/phi4mini_summarized_only_lora/phi4mini_summarized_only_lora/checkpoint-2000

USING MODEL_DIR: /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora
adapter_config exists: True


In [4]:
# CELL 4 — Helper functions

def fmt_time(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def gpu_report(title="GPU"):
    print("\n" + title)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
        print("Reserved :", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")
    else:
        print("CUDA not available")

def get_compute_dtype():
    return torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

def single_gpu_device_map():
    return {"": 0} if torch.cuda.is_available() else None

def build_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=get_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )

def manual_train_test_split(df, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(df))
    rng.shuffle(idx)
    n_test = int(round(len(df) * test_size))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

gpu_report('GPU status')



GPU status
GPU: Tesla T4
Allocated: 0.0 GB
Reserved : 0.0 GB


In [5]:
# CELL 5 — Load dataset and create test_df

def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.DataFrame([data])
        raise ValueError("Unsupported JSON format")
    except json.JSONDecodeError:
        return pd.read_json(path, lines=True)

_df = load_json(DATA_PATH)
print(f"Loaded: {len(_df):,} rows | columns: {list(_df.columns)}")

required = ["question", "right_answer"]
missing = [c for c in required if c not in _df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# summarized datasets sometimes use selected_evidence; original may use knowledge
if "selected_evidence" in _df.columns:
    context_col = "selected_evidence"
elif "knowledge" in _df.columns:
    context_col = "knowledge"
elif "context" in _df.columns:
    context_col = "context"
else:
    raise ValueError("Need one context column: selected_evidence, knowledge, or context")

_df["_context"] = _df[context_col].fillna("").astype(str)
_df["question"] = _df["question"].fillna("").astype(str)
_df["right_answer"] = _df["right_answer"].fillna("").astype(str)
_df = _df[(_df["_context"].str.strip() != "") & (_df["question"].str.strip() != "") & (_df["right_answer"].str.strip() != "")].reset_index(drop=True)

if SAMPLE_SIZE and len(_df) > SAMPLE_SIZE:
    idx = np.random.choice(len(_df), SAMPLE_SIZE, replace=False)
    idx.sort()
    _df = _df.iloc[idx].reset_index(drop=True)

train_df, test_df = manual_train_test_split(_df, test_size=TEST_RATIO, seed=SEED)
print(f"Usable rows: {len(_df):,}")
print(f"Context column: {context_col}")
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")


Loaded: 10,000 rows | columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer', 'original_knowledge', 'selected_evidence', 'abstractive_summary', 'summary_method', 'selection_fallback_to_original', 'summary_fallback_to_selected_evidence', 'answer_preserved_in_final_summary', 'original_char_len', 'selected_char_len', 'summary_char_len', 'num_original_sentences', 'num_selected_sentences', 'selected_answer_preserved']
Usable rows: 10,000
Context column: selected_evidence
Train: 8,000 | Test: 2,000


In [6]:
# CELL 6 — Prompt template
ORIG_SYSTEM = """You are an accurate question-answering assistant.
Answer the question based solely on the provided context.
If the answer is not in the context, reply: I don't know.
Be concise."""

SUMM_SYSTEM = """You are a precise, context-grounded question-answering assistant.
The context is answer-focused: irrelevant sentences have been removed.
Rules:
1. Answer ONLY from the context.
2. If not supported, reply exactly: I don't know.
3. Keep the answer short and direct.
4. Do NOT fabricate information."""

SYSTEM_PROMPT = SUMM_SYSTEM if RUN_TYPE == "Summarized Only" else ORIG_SYSTEM

def build_input_prompt(system: str, context: str, question: str) -> str:
    return (f'<|system|>\n{system}\n<|end|>\n'
            f'<|user|>\nContext:\n{context.strip()}\n\nQuestion: {question.strip()}\n<|end|>\n'
            f'<|assistant|>\n')

print('Prompt ready.')


Prompt ready.


In [7]:
# CELL 7 — Load base Phi-4-mini for inference

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Loading base model:', MODEL_ID)
bnb_inf = build_bnb_config() if USE_4BIT else None

base_kwargs = dict(
    trust_remote_code=False,
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
    torch_dtype=get_compute_dtype(),
)
if USE_4BIT:
    base_kwargs.update(
        quantization_config=bnb_inf,
        device_map=single_gpu_device_map(),
    )

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **base_kwargs)
base_model.config.use_cache = True

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False, use_fast=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = 'left'

print('Base model loaded.')
gpu_report('GPU after base load')


Loading base model: microsoft/Phi-4-mini-instruct


[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Base model loaded.

GPU after base load
GPU: Tesla T4
Allocated: 2.69 GB
Reserved : 2.88 GB


In [8]:
# CELL 8 — Metrics functions without sklearn
print(f'Loading semantic model: {SEM_MODEL_ID}')
sem_model = SentenceTransformer(SEM_MODEL_ID)
print('Semantic model loaded.')

def normalize_answer(s):
    s = str(s).lower().strip()
    s = re.sub(r'[^\w\s]', '', s)
    stopwords = {'the','a','an','is','are','was','were','of','in','on','at','to','for','with','by'}
    return ' '.join([t for t in s.split() if t not in stopwords])

def exact_match(p, g):
    return float(normalize_answer(p) == normalize_answer(g))

def token_f1(p, g):
    pt = normalize_answer(p).split()
    gt = normalize_answer(g).split()
    if not pt or not gt:
        return 0.0
    com = Counter(pt) & Counter(gt)
    nc = sum(com.values())
    if nc == 0:
        return 0.0
    prec = nc / len(pt)
    rec = nc / len(gt)
    return 2 * prec * rec / (prec + rec)

def fuzzy_score(p, g):
    return rfuzz.token_sort_ratio(str(p).lower(), str(g).lower()) / 100.0

def sem_sim_batch(preds, golds):
    ep = sem_model.encode(preds, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    eg = sem_model.encode(golds, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    return np.sum(ep * eg, axis=1).tolist()

def answer_in_context(pred, context):
    p = normalize_answer(pred)
    c = normalize_answer(context)
    return float(bool(p) and p in c)

def context_util(pred, context):
    pt = set(normalize_answer(pred).split())
    ct = set(normalize_answer(context).split())
    if not pt:
        return 0.0
    return len(pt & ct) / len(pt)

def evaluate_all(preds, golds, contexts):
    em = [exact_match(p, g) for p, g in zip(preds, golds)]
    f1 = [token_f1(p, g) for p, g in zip(preds, golds)]
    fuz = [fuzzy_score(p, g) for p, g in zip(preds, golds)]
    sem = sem_sim_batch(preds, golds)
    aic = [answer_in_context(p, c) for p, c in zip(preds, contexts)]
    cut = [context_util(p, c) for p, c in zip(preds, contexts)]
    agg = [(0.35*f1[i] + 0.25*fuz[i] + 0.30*sem[i] + 0.10*aic[i]) for i in range(len(preds))]
    hal = [float(a < 0.5) for a in agg]
    return {
        'accuracy_threshold_50': float(np.mean([a >= 0.5 for a in agg])),
        'exact_match': float(np.mean(em)),
        'mean_token_f1': float(np.mean(f1)),
        'mean_fuzzy': float(np.mean(fuz)),
        'mean_aggregate_score': float(np.mean(agg)),
        'answer_in_context_rate': float(np.mean(aic)),
        'hallucination_rate': float(np.mean(hal)),
        'mean_semantic_similarity': float(np.mean(sem)),
        '_em': em, '_f1': f1, '_fuz': fuz, '_sem': sem, '_agg': agg, '_aic': aic, '_hal': hal, '_cut': cut
    }


Loading semantic model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic model loaded.


In [9]:
# CELL 9 — Inference functions
GEN_PARAMS = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=1.0,
    repetition_penalty=1.05,
    pad_token_id=base_tokenizer.pad_token_id,
    eos_token_id=base_tokenizer.eos_token_id,
)

def extract_answer(full_text, prompt):
    if full_text.startswith(prompt):
        ans = full_text[len(prompt):]
    elif '<|assistant|>' in full_text:
        ans = full_text.split('<|assistant|>')[-1]
    else:
        ans = full_text

    stop_markers = ['<|user|>', '<|system|>', '<|assistant|>', '<|end|>', '<|endoftext|>', '\nContext:', '\nQuestion:', 'Context:', 'Question:']
    for marker in stop_markers:
        if marker in ans:
            ans = ans.split(marker)[0]
    ans = re.sub(r'^(answer\s*:\s*)', '', ans, flags=re.I).strip()
    ans = ans.strip(' \n\t:.-')
    lines = [l.strip() for l in ans.split('\n') if l.strip()]
    return lines[0] if lines else ""

@torch.no_grad()
def run_inference(model, tokenizer, df, system_prompt, context_col='_context', desc='Inference'):
    model.eval()
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        prompt = build_input_prompt(system_prompt, row[context_col], row['question'])
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(model.device)
        out = model.generate(**inputs, **GEN_PARAMS)
        text = tokenizer.decode(out[0], skip_special_tokens=False)
        preds.append(extract_answer(text, prompt))
    return preds

print('Inference functions ready.')


Inference functions ready.


In [10]:
# CELL 10 — Load LoRA adapter from working and run inference
print('=' * 60)
print(f'INFERENCE: Fine-Tuned Phi-4-mini + {RUN_TYPE} LoRA')
print('=' * 60)
print('Loading adapter:', MODEL_DIR)

model_inf = PeftModel.from_pretrained(base_model, str(MODEL_DIR))

t0 = time.time()
preds = run_inference(
    model_inf, base_tokenizer, test_df,
    SYSTEM_PROMPT, '_context',
    desc=f'{RUN_TYPE} LoRA Inference'
)
inference_time = time.time() - t0
print(f'Inference time: {fmt_time(inference_time)}')

for i in range(min(3, len(preds))):
    print(f'\n--- Example {i+1} ---')
    print(f'Q   : {test_df["question"].iloc[i]}')
    print(f'Ref : {test_df["right_answer"].iloc[i]}')
    print(f'Pred: {preds[i]}')

model_inf = model_inf.unload()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


INFERENCE: Fine-Tuned Phi-4-mini + Summarized Only LoRA
Loading adapter: /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora


Summarized Only LoRA Inference:   0%|          | 0/2000 [00:00<?, ?it/s]

Inference time: 01:19:40

--- Example 1 ---
Q   : Simon WIlliam Gamble played non league football for what club located in Lincolnshire, England?
Ref : Grantham Town Football Club
Pred: Grantham Town Football Club

--- Example 2 ---
Q   : Both Janggi and Isola are board-based strategy games; however, which game is played on a 9x10 gameboard similar to Chinese Chess?
Ref : Janggi
Pred: Janggi

--- Example 3 ---
Q   : Where was the film A Taxi Driver released?
Ref : South Korea
Pred: South Korea


In [11]:
# CELL 11 — Evaluate + save CSV outputs
golds = list(test_df['right_answer'])
contexts = list(test_df['_context'])

print(f'Evaluating {RUN_TYPE} model...')
eval_start = time.time()
metrics = evaluate_all(preds, golds, contexts)
print(f'Evaluation time: {fmt_time(time.time() - eval_start)}')

DISPLAY_METRIC_MAP = {
    'accuracy_by_threshold': 'accuracy_threshold_50',
    'exact_match': 'exact_match',
    'mean_token_f1': 'mean_token_f1',
    'mean_fuzzy': 'mean_fuzzy',
    'mean_aggregate': 'mean_aggregate_score',
    'answer_in_context': 'answer_in_context_rate',
    'hallucination_rate': 'hallucination_rate',
    'mean_semantic_similarity': 'mean_semantic_similarity',
}

metric_rows = [{'metric': shown, 'value': round(metrics[key], 6)} for shown, key in DISPLAY_METRIC_MAP.items()]
df_metrics = pd.DataFrame(metric_rows)
print(df_metrics.to_string(index=False))

pred_df = test_df[['question', 'right_answer']].copy()
pred_df['predicted_answer'] = preds
pred_df['exact_match'] = metrics['_em']
pred_df['token_f1'] = metrics['_f1']
pred_df['fuzzy_score'] = metrics['_fuz']
pred_df['semantic_sim'] = metrics['_sem']
pred_df['aggregate_score'] = metrics['_agg']
pred_df['answer_in_context'] = metrics['_aic']
pred_df['hallucinated'] = metrics['_hal']
pred_df['context_util'] = metrics['_cut']
pred_df['method'] = RUN_TYPE.lower().replace(' ', '_')

prefix = RUN_TYPE.lower().replace(' ', '_')
predictions_csv = OUTPUT_DIR / f"{prefix}_predictions.csv"
metrics_csv = OUTPUT_DIR / f"{prefix}_metrics.csv"
summary_txt = OUTPUT_DIR / f"{prefix}_summary.txt"

pred_df.to_csv(predictions_csv, index=False)
df_metrics.to_csv(metrics_csv, index=False)

summary = f"""
========================================================
FINE-TUNED PHI-4-MINI RAG - {RUN_TYPE.upper()}
========================================================
Dataset     : {len(test_df)} test examples
Model       : {MODEL_ID}
Adapter     : {MODEL_DIR}
Seed        : {SEED}
Infer time  : {fmt_time(inference_time)}

--- METRICS ---
{df_metrics.to_string(index=False)}

--- SAVED FILES ---
Predictions CSV: {predictions_csv}
Metrics CSV    : {metrics_csv}
Summary TXT    : {summary_txt}
""".strip()

summary_txt.write_text(summary, encoding='utf-8')
print('\n' + summary)


Evaluating Summarized Only model...
Evaluation time: 00:00:00
                  metric    value
   accuracy_by_threshold 0.827500
             exact_match 0.724000
           mean_token_f1 0.795369
              mean_fuzzy 0.824127
          mean_aggregate 0.832754
       answer_in_context 0.914000
      hallucination_rate 0.172500
mean_semantic_similarity 0.856476

FINE-TUNED PHI-4-MINI RAG - SUMMARIZED ONLY
Dataset     : 2000 test examples
Model       : microsoft/Phi-4-mini-instruct
Adapter     : /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora
Seed        : 42
Infer time  : 01:19:40

--- METRICS ---
                  metric    value
   accuracy_by_threshold 0.827500
             exact_match 0.724000
           mean_token_f1 0.795369
              mean_fuzzy 0.824127
          mean_aggregate 0.832754
       answer_in_context 0.914000
      hallucination_rate 0.172500
mean_semantic_similarity 0.856476

--- SAVED FILES ---
Predict

In [12]:
# CELL 12 — Show saved outputs

def folder_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())
    return round(total / 1024 / 1024, 2)

print('\n' + '='*70)
print('DONE — EVALUATION OUTPUTS')
print('='*70)
print(f'LoRA folder   : {MODEL_DIR}')
print(f'Folder size   : {folder_size_mb(MODEL_DIR):.2f} MB')
print(f'Inference time: {fmt_time(inference_time)}')
print(f'Output dir    : {OUTPUT_DIR}')
print('\nFiles:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        print('-', f)



DONE — EVALUATION OUTPUTS
LoRA folder   : /kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models/phi4mini_summarized_only_lora
Folder size   : 375.05 MB
Inference time: 01:19:40
Output dir    : /kaggle/working/PHI4_EVAL_ONLY

Files:
- /kaggle/working/PHI4_EVAL_ONLY/summarized_only_metrics.csv
- /kaggle/working/PHI4_EVAL_ONLY/summarized_only_predictions.csv
- /kaggle/working/PHI4_EVAL_ONLY/summarized_only_summary.txt


In [15]:
import shutil

shutil.make_archive(
    "/kaggle/working/phi4mini_summarized_only_lora",
    "zip",
    "/kaggle/input/datasets/belalhesham/qa-summ/PHI4_SUMMARIZED_ONLY_RAG/models",
    "phi4mini_summarized_only_lora"
)

print("ZIP READY")

ZIP READY


In [13]:
import shutil

src = "/kaggle/input/qa_summ/PHI4_SUMMARIZED_ONLY_RAG/phi4mini_summarized_only_lora/phi4mini_summarized_only_lora"
dst = "/kaggle/working/phi4mini_summarized_only_lora"

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Done")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/qa_summ/PHI4_SUMMARIZED_ONLY_RAG/phi4mini_summarized_only_lora/phi4mini_summarized_only_lora'